In [0]:
%run ../utils/init.py

In [0]:
files = dbutils.fs.ls(f"{RAW}/osm/")
for f in files:
    print(f.name, f.size)

In [0]:
df = spark.read.parquet(f"{RAW}/osm/poi_commerce.parquet")
df.printSchema()
df.show(2)

In [0]:
from functools import reduce

categories = ["commerce", "education", "parcs", 
              "restauration", "sante", "services", "transport"]

dfs = [spark.read.parquet(f"{RAW}/osm/poi_{cat}.parquet") 
       for cat in categories]

df_raw = reduce(lambda a, b: a.union(b), dfs)

print(f"Total POI : {df_raw.count()}")
df_raw.groupBy("category").count().orderBy("count", ascending=False).show()

In [0]:
df_bronze = (df_raw
    # Supprimer les POI sans coordonnées
    .filter(F.col("latitude").isNotNull() & F.col("longitude").isNotNull())
    # Supprimer les doublons sur osm_id
    .dropDuplicates(["osm_id"])
    # Signal gentrification bio/bobo
    .withColumn("is_bio_bobo",
        F.col("shop").isin("organic", "deli") |
        F.lower(F.col("name")).rlike("bio|vegan|végétalien|zéro.déchet|zero.dechet")
    )
    .withColumn("ingestion_timestamp", F.current_timestamp())
)

print(f"Lignes après nettoyage : {df_bronze.count()}")

df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("category") \
    .save(f"{BRONZE}/osm_poi/")

print("✅ Bronze OSM POI écrit")